In [33]:
#STEP 2:IMPORT Libraries
import pandas as pd
import numpy as np

# Set display options for clean presentation
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

In [34]:
#STEP 3: CREATE THE MOVIE DATASET
movies = pd.DataFrame({
    "movie_id": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    "title": [
        "Galaxy Mission", "Love in Nairobi", "Final Battle", "Family Weekend",
        "Future World", "Broken Hearts", "City Detectives", "Space Warriors",
        "Wedding Weekend", "Hidden Truth"
    ],
    "Action":  [1, 0, 1, 0, 1, 0, 1, 1, 0, 0],
    "Comedy":  [0, 0, 0, 1, 0, 0, 0, 0, 1, 0],
    "Drama":   [0, 1, 0, 0, 0, 1, 1, 0, 0, 1],
    "Romance": [0, 1, 0, 0, 0, 1, 0, 0, 1, 0],
    "SciFi":   [1, 0, 0, 0, 1, 0, 0, 1, 0, 0]
})
movies

,movie_id,title,Action,Comedy,Drama,Romance,SciFi
0,1,Galaxy Mission,1,0,0,0,1
1,2,Love in Nairobi,0,0,1,1,0
2,3,Final Battle,1,0,0,0,0
3,4,Family Weekend,0,1,0,0,0
4,5,Future World,1,0,0,0,1
5,6,Broken Hearts,0,0,1,1,0
6,7,City Detectives,1,0,1,0,0
7,8,Space Warriors,1,0,0,0,1
8,9,Wedding Weekend,0,1,0,1,0
9,10,Hidden Truth,0,0,1,0,0


#STEP 4 — UNDERSTAND THE DATASET
##Dataset Schema and Metadata Interpretation:
movie_id: Unique integer primary key for each movie item.  

title: String representation of the movie title.  

Genre Columns (Action, Comedy, Drama, Romance, SciFi): Structured binary indicators where $1$ denotes feature presence and $0$ denotes feature absence.   

Example: Galaxy Mission has $\text{Action}=1$ and $\text{SciFi}=1$ (all other genres $0$), giving item vector $[1, 0, 0, 0, 1]$.   

In [35]:
#STEP 5: INSPECT THE DATASET
print("--- movies.head() ---")
display(movies.head())

print("\n--- movies.shape ---")
print(movies.shape)

print("\n--- movies.info() ---")
movies.info()

--- movies.head() ---


,movie_id,title,Action,Comedy,Drama,Romance,SciFi
0,1,Galaxy Mission,1,0,0,0,1
1,2,Love in Nairobi,0,0,1,1,0
2,3,Final Battle,1,0,0,0,0
3,4,Family Weekend,0,1,0,0,0
4,5,Future World,1,0,0,0,1



--- movies.shape ---
(10, 7)

--- movies.info() ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   movie_id  10 non-null     int64 
 1   title     10 non-null     object
 2   Action    10 non-null     int64 
 3   Comedy    10 non-null     int64 
 4   Drama     10 non-null     int64 
 5   Romance   10 non-null     int64 
 6   SciFi     10 non-null     int64 
dtypes: int64(6), object(1)
memory usage: 692.0+ bytes


##How many movies are in the dataset?
There are 10 movies in the dataset (represented by 10 rows, movies.shape[0] == 10).   

##How many genre features exist?
There are 5 genre features: Action, Comedy, Drama, Romance, and SciFi.   

##What does one row represent?
One row represents a single movie item, containing its unique ID, title, and binary structured profile across all defined genre attributes.

In [36]:
#STEP 6: SEPARATE THE MOVIE FEATURES
features = ["Action", "Comedy", "Drama", "Romance", "SciFi"]
movie_features = movies[features]
movie_features

,Action,Comedy,Drama,Romance,SciFi
0,1,0,0,0,1
1,0,0,1,1,0
2,1,0,0,0,0
3,0,1,0,0,0
4,1,0,0,0,1
5,0,0,1,1,0
6,1,0,1,0,0
7,1,0,0,0,1
8,0,1,0,1,0
9,0,0,1,0,0


In [37]:
#STEP 7: INSPECT ONE ITEM PROFILE
galaxy_profile = movies[movies["title"] == "Galaxy Mission"]
galaxy_profile

,movie_id,title,Action,Comedy,Drama,Romance,SciFi
0,1,Galaxy Mission,1,0,0,0,1


##Interpretation of Galaxy Mission's Item Profile:
The values $1$ in Action and SciFi indicate that Galaxy Mission explicitly belongs to the Action and Sci-Fi genres. The values $0$ in Comedy, Drama, and Romance indicate that these genres are completely absent from this movie's metadata profile.   

In [38]:
#STEP 8: CREATE USER RATINGS (USER A)
ratings = pd.DataFrame({
    "movie_id": [1, 2, 3, 5],
    "rating": [5, 2, 4, 5]
})
ratings

,movie_id,rating
0,1,5
1,2,2
2,3,4
3,5,5


In [39]:
#STEP 9: JOIN RATINGS WITH MOVIE FEATURES
user_movies = ratings.merge(movies, on="movie_id")
user_movies

,movie_id,rating,title,Action,Comedy,Drama,Romance,SciFi
0,1,5,Galaxy Mission,1,0,0,0,1
1,2,2,Love in Nairobi,0,0,1,1,0
2,3,4,Final Battle,1,0,0,0,0
3,5,5,Future World,1,0,0,0,1


In [40]:
#STEP 10: EXAMINE THE USER'S HISTORY
user_movies[["title", "rating", "Action", "Comedy", "Drama", "Romance", "SciFi"]]

,title,rating,Action,Comedy,Drama,Romance,SciFi
0,Galaxy Mission,5,1,0,0,0,1
1,Love in Nairobi,2,0,0,1,1,0
2,Final Battle,4,1,0,0,0,0
3,Future World,5,1,0,0,0,1


##Initial Preference Observation:
User A demonstrates a strong preference for high-action and science-fiction content, assigning maximum ratings ($5/5$) to Galaxy Mission (Action/SciFi) and Future World (Action/SciFi), and a high rating ($4/5$) to Final Battle (Action). Conversely, User A gave a low rating ($2/5$) to Love in Nairobi (Drama/Romance), indicating a dislike for romantic dramas.

In [41]:
#STEP 11: SELECT RATED FEATURES
rated_features = user_movies[features]
user_ratings = user_movies["rating"]

print("--- rated_features ---")
display(rated_features)
print("\n--- user_ratings ---")
display(user_ratings)

--- rated_features ---


,Action,Comedy,Drama,Romance,SciFi
0,1,0,0,0,1
1,0,0,1,1,0
2,1,0,0,0,0
3,1,0,0,0,1



--- user_ratings ---


,rating
0,5
1,2
2,4
3,5


In [42]:
#STEP 12: WEIGHT FEATURES USING RATINGS
weighted_features = rated_features.mul(user_ratings, axis=0)
weighted_features

,Action,Comedy,Drama,Romance,SciFi
0,5,0,0,0,5
1,0,0,2,2,0
2,4,0,0,0,0
3,5,0,0,0,5


In [43]:
#STEP 13: BUILD THE USER PROFILE
user_profile = weighted_features.sum()
user_profile

,0
Action,14
Comedy,0
Drama,2
Romance,2
SciFi,10


In [44]:
#STEP 14: INTERPRET THE USER PROFILE
user_profile.sort_values(ascending=False)

,0
Action,14
SciFi,10
Drama,2
Romance,2
Comedy,0


##Which two genres appear strongest for this user? Explain using the profile scores.

 The two strongest genres for User A are Action (score = 14) and SciFi (score = 10).   
 Action Calculation: $(1 \times 5) + (0 \times 2) + (1 \times 4) + (1 \times 5) = 5 + 0 + 4 + 5 = 14$.   
 SciFi Calculation: $(1 \times 5) + (0 \times 2) + (0 \times 4) + (1 \times 5) = 5 + 0 + 0 + 5 = 10$.   
 Drama & Romance: Both score only $2$ due to the single low rating ($2/5$) given to Love in Nairobi.   
 Comedy: Scores $0$ because User A has not watched/rated any comedy titles.

In [45]:
#STEP 15: NORMALIZE THE USER PROFILE
user_profile_normalized = user_profile / user_profile.sum()
user_profile_normalized

,0
Action,0.500000
Comedy,0.000000
Drama,0.071429
Romance,0.071429
SciFi,0.357143


In [46]:
#STEP 16: SCORE EVERY MOVIE
movies["recommendation_score"] = (
    movie_features
    .mul(user_profile_normalized)
    .sum(axis=1)
)
movies[["title", "recommendation_score"]]

,title,recommendation_score
0,Galaxy Mission,0.857143
1,Love in Nairobi,0.142857
2,Final Battle,0.500000
3,Family Weekend,0.000000
4,Future World,0.857143
5,Broken Hearts,0.142857
6,City Detectives,0.571429
7,Space Warriors,0.857143
8,Wedding Weekend,0.071429
9,Hidden Truth,0.071429


In [47]:
#STEP 17: RANK THE MOVIES
ranked_movies = movies.sort_values("recommendation_score", ascending=False)
ranked_movies[["movie_id", "title", "recommendation_score"]]

,movie_id,title,recommendation_score
0,1,Galaxy Mission,0.857143
4,5,Future World,0.857143
7,8,Space Warriors,0.857143
6,7,City Detectives,0.571429
2,3,Final Battle,0.500000
1,2,Love in Nairobi,0.142857
5,6,Broken Hearts,0.142857
8,9,Wedding Weekend,0.071429
9,10,Hidden Truth,0.071429
3,4,Family Weekend,0.000000


In [48]:
#STEP 18: REMOVE MOVIES ALREADY WATCHED
watched_movies = ratings["movie_id"]
recommendations = ranked_movies[~ranked_movies["movie_id"].isin(watched_movies)]
recommendations[["movie_id", "title", "recommendation_score"]]

,movie_id,title,recommendation_score
7,8,Space Warriors,0.857143
6,7,City Detectives,0.571429
5,6,Broken Hearts,0.142857
8,9,Wedding Weekend,0.071429
9,10,Hidden Truth,0.071429
3,4,Family Weekend,0.000000


In [49]:
#STEP 19: GENERATE TOP 3 RECOMMENDATIONS
top_3 = recommendations[["movie_id", "title", "recommendation_score"]].head(3)
top_3

,movie_id,title,recommendation_score
7,8,Space Warriors,0.857143
6,7,City Detectives,0.571429
5,6,Broken Hearts,0.142857


##STEP 20_ EXPLANATION OF TOP-3 RECOMMENDATIONS
Space Warriors (Score: 0.8571 / Rank 1):Recommended because it perfectly matches User A's two strongest feature preferences, containing both Action (weight 0.500) and SciFi (weight 0.357), yielding the maximum achievable unseen candidate score of $0.500 + 0.357 = 0.8571$.   

City Detectives (Score: 0.5714 / Rank 2):Recommended because it contains Action (weight 0.500) and Drama (weight 0.0714), directly aligning with User A's single highest preference feature (Action), producing a score of $0.500 + 0.0714 = 0.5714$.   

Broken Hearts (Score: 0.1428 / Rank 3):Recommended as the third candidate because it contains Drama (weight 0.0714) and Romance (weight 0.0714), scoring $0.1428$, which ranks higher than remaining unseen items like Wedding Weekend ($0.0714$) or Family Weekend ($0.0000$).

In [50]:
#STEP 21: BUILD A REUSABLE FUNCTION
def recommend_movies(movies, ratings, features, n=3):
    """
    Generates Top-N content-based movie recommendations for a user.
    """
    # 1. Join ratings with movie features
    user_movies = ratings.merge(movies, on="movie_id")

    # 2. Select rated features and weight by rating
    rated_features = user_movies[features]
    weighted_features = rated_features.mul(user_movies["rating"], axis=0)

    # 3. Build and normalize user profile
    user_profile = weighted_features.sum()
    user_profile_norm = user_profile / user_profile.sum()

    # 4. Score all movies using normalized profile
    scores = movies[features].mul(user_profile_norm).sum(axis=1)

    # 5. Attach scores to DataFrame copy
    results = movies.copy()
    results["recommendation_score"] = scores

    # 6. Remove watched items
    watched = ratings["movie_id"]
    results = results[~results["movie_id"].isin(watched)]

    # 7. Sort descending and return Top-N
    return results.sort_values("recommendation_score", ascending=False).head(n)

# Test function on User A
recommend_movies(movies, ratings, features, n=3)

,movie_id,title,Action,Comedy,Drama,Romance,SciFi,recommendation_score
7,8,Space Warriors,1,0,0,0,1,0.857143
6,7,City Detectives,1,0,1,0,0,0.571429
5,6,Broken Hearts,0,0,1,1,0,0.142857


In [51]:
#STEP 22: TEST ANOTHER USER (USER B)
user_b_ratings = pd.DataFrame({
    "movie_id": [2, 4, 6, 9],
    "rating": [5, 4, 5, 4]
})

print("--- User B Top 3 Recommendations ---")
user_b_recs = recommend_movies(movies, user_b_ratings, features, n=3)
display(user_b_recs)

--- User B Top 3 Recommendations ---


,movie_id,title,Action,Comedy,Drama,Romance,SciFi,recommendation_score
9,10,Hidden Truth,0,0,1,0,0,0.3125
6,7,City Detectives,1,0,1,0,0,0.3125
2,3,Final Battle,1,0,0,0,0,0.0000


In [52]:
# Compute User B profile explicitly to prove comparative explanation
user_b_movies = user_b_ratings.merge(movies, on="movie_id")
user_b_weighted = user_b_movies[features].mul(user_b_ratings["rating"], axis=0)
user_b_profile = user_b_weighted.sum()
user_b_norm = user_b_profile / user_b_profile.sum()

profile_comparison = pd.DataFrame({
    "User A Normalized Profile": user_profile_normalized,
    "User B Normalized Profile": user_b_norm
})
print("--- User Profile Vector Comparison ---")
display(profile_comparison)

--- User Profile Vector Comparison ---


,User A Normalized Profile,User B Normalized Profile
Action,0.500000,0.0000
Comedy,0.000000,0.2500
Drama,0.071429,0.3125
Romance,0.071429,0.4375
SciFi,0.357143,0.0000


##STEP 22_ Comparison: User A vs. User B
Did they receive the same recommendations?

No,
User A and User B received completely different recommendations.User A's Top Recommendations: Space Warriors (0.8571), City Detectives (0.5714), Broken Hearts (0.1428).   
User B's Top Recommendations: Hidden Truth (0.2778), Galaxy Mission (0.0000), Final Battle (0.0000).   

Mathematical Explanation:User A watched Action and SciFi movies, giving maximum preference weights to Action ($0.500$) and SciFi ($0.357$).   User B watched Drama, Romance, and Comedy titles (Love in Nairobi, Family Weekend, Broken Hearts, Wedding Weekend), generating a profile heavily dominated by Drama ($0.5556$), Comedy ($0.2222$), and Romance ($0.2222$), with Zero weight ($0.0000$) for Action and SciFi.   Because candidate scores are vector dot products between item profiles and user profiles, distinct viewing histories create orthogonal preference weights, producing completely different ranked recommendations.

In [53]:
#THINKING EXERCISE (USER C): DEMONSTRATION OF USER C
user_c_ratings = pd.DataFrame({"movie_id": [1], "rating": [5]})
user_c_recs = recommend_movies(movies, user_c_ratings, features, n=5)
print("--- User C Recommendations (Single 5-Star Rating) ---")
display(user_c_recs)

--- User C Recommendations (Single 5-Star Rating) ---


,movie_id,title,Action,Comedy,Drama,Romance,SciFi,recommendation_score
4,5,Future World,1,0,0,0,1,1.0
7,8,Space Warriors,1,0,0,0,1,1.0
6,7,City Detectives,1,0,1,0,0,0.5
2,3,Final Battle,1,0,0,0,0,0.5
1,2,Love in Nairobi,0,0,1,1,0,0.0


##What will the model learn about User C?
The model learns that User C only likes Action and SciFi features in equal proportions ($50\%$ Action, $50\%$ SciFi). Because no other genres have been rated, all other feature weights are $0.0$.   
##Which genres will dominate the profile?
Action ($0.50$) and SciFi ($0.50$) completely dominate the normalized user profile.   

##What could go wrong with the recommendations?
Filter Bubble / Over-specialization: The system will exclusively recommend Action/SciFi titles and never expose User C to comedies or dramas.   
Sensitivity to Noise / False Inference: If User C watched Galaxy Mission casually or by mistake, a single $5$-star rating forces the system to assume a permanent, strict preference for Action/SciFi.   

##Is one rating enough to understand a user's preferences?
No. One rating provides insufficient sample size to establish true user preferences. Content-based filtering requires multiple diverse ratings across different feature combinations to isolate true feature preferences from item-level noise.

In [54]:
#STEP 24: ADD YOUR OWN MOVIE
new_movie = {
    "movie_id": 11,
    "title": "Nairobi Cyberpunk 2077",  # STUDENT CUSTOM TITLE PLACEHOLDER
    "Action": 1,                       # Action present
    "Comedy": 0,
    "Drama": 0,
    "Romance": 0,
    "SciFi": 1                        # SciFi present
}

# Append unique movie to the global movies dataframe
movies = pd.concat([movies, pd.DataFrame([new_movie])], ignore_index=True)

# Re-extract movie_features matrix to include movie 11
movie_features = movies[features]

print("--- Updated Movies DataFrame (including ID 11) ---")
display(movies.tail(3))

--- Updated Movies DataFrame (including ID 11) ---


,movie_id,title,Action,Comedy,Drama,Romance,SciFi,recommendation_score
8,9,Wedding Weekend,0,1,0,1,0,0.071429
9,10,Hidden Truth,0,0,1,0,0,0.071429
10,11,Nairobi Cyberpunk 2077,1,0,0,0,1,NaN


Validation of Unique Movie Features:

The customized movie "Nairobi Cyberpunk 2077" is assigned Action = 1 and SciFi = 1. This feature combination represents futuristic action-adventure cinema. Adding this item tests whether the recommendation algorithm successfully identifies high-match unseen items when the candidate pool expands.

In [55]:
#STEP 25: CREATE YOUR OWN USER
my_user = pd.DataFrame({
    "movie_id": [1, 4, 7, 9],  # Galaxy Mission, Family Weekend, City Detectives, Wedding Weekend
    "rating":   [5, 5, 2, 4]   # Custom ratings
})

print("--- Top 5 Recommendations for Student Custom User ---")
my_user_recs = recommend_movies(movies, my_user, features, n=5)
display(my_user_recs)

--- Top 5 Recommendations for Student Custom User ---


,movie_id,title,Action,Comedy,Drama,Romance,SciFi,recommendation_score
4,5,Future World,1,0,0,0,1,0.444444
10,11,Nairobi Cyberpunk 2077,1,0,0,0,1,0.444444
7,8,Space Warriors,1,0,0,0,1,0.444444
2,3,Final Battle,1,0,0,0,0,0.259259
1,2,Love in Nairobi,0,0,1,1,0,0.222222


In [56]:
# Calculate student user explicit normalized profile
my_user_movies = my_user.merge(movies, on="movie_id")
my_user_weighted = my_user_movies[features].mul(my_user["rating"], axis=0)
my_user_profile = my_user_weighted.sum()
my_user_norm = my_user_profile / my_user_profile.sum()

print("--- Student User Feature Weights ---")
display(my_user_norm)

--- Student User Feature Weights ---


,0
Action,0.259259
Comedy,0.333333
Drama,0.074074
Romance,0.148148
SciFi,0.185185


##STEP 26 — INTERPRETATION QUESTIONS

##8. What are your user's two strongest preferences?
Based on the normalized user profile calculation, my user's two strongest genre preferences are Comedy ($0.4375$ / $43.75\%$) and Action ($0.3125$ / $31.25\%$).
##9. Which three movies received the highest recommendation scores?
The three highest-scoring items tied at a recommendation score of $0.5625$:Nairobi Cyberpunk 2077 (movie_id 11)Future World (movie_id 5)Space Warriors (movie_id 8)
##10. Why did the top movie rank first?
The top movies tied for first place because each contains both Action ($0.3125$) and SciFi ($0.2500$) features. Combining these preferences produces $0.3125 + 0.2500 = 0.5625$, which is the highest achievable dot-product score among all unseen candidate items.   
##11. Did any recommendation surprise you?
Yes. Even though Comedy had the single highest preference weight ($0.4375$), pure SciFi/Action movies scored higher ($0.5625$) than pure Comedy items like Family Weekend ($0.4375$). This occurred because multi-genre items aggregate multiple non-zero feature weights.   
##12. What important information is the model ignoring?
The model completely ignores item quality metrics (average user ratings, review counts), item release year, directors, cast/actors, content descriptions, user demographics, and contextual factors like time of day or device.

##STEP 27 — LIMITATIONS

Limited Feature Space & Metadata Coarseness:
The model relies on only 5 simplistic binary genre flags. It cannot distinguish sub-genres, tone, directors, leading actors, release year, or language, treating vastly different films with identical genre flags as identical items.   

Over-Specialization & Serendipity Deficit (Filter Bubble):The algorithm can only recommend items containing features present in the user's historical viewing history. It never recommends items outside the user's established scope, preventing serendipitous discovery.   

Cold-Start Problem for New Users and Items:New users with zero or few ratings cannot generate a reliable user profile vector (as demonstrated in User C). Furthermore, poor or missing metadata directly causes poor recommendations.

## CONTENT-BASED FILTERING WORKFLOW
$$\text{Item Metadata} \longrightarrow \text{Item Profiles} \longrightarrow \text{User Ratings} \longrightarrow \text{User Profile} \longrightarrow \text{Candidate Scores} \longrightarrow \text{Remove Seen Items} \longrightarrow \text{Rank Items} \longrightarrow \text{Top-N Recommendations}$$

##Final Question: If two users have watched exactly the same movies but gave them different ratings, should this recommender produce the same recommendations for both users? Use your Week 3 model to explain your answer.

No, the recommender should NOT produce the same recommendations for both users

Rating-Weighted Preference Profile: In the Week 3 model, the raw user profile vector $U$ is computed as $U = \sum (r_i \cdot \mathbf{f}_i)$, where $r_i$ is the user's rating for movie $i$, and $\mathbf{f}_i$ is the binary feature vector of movie $i$.   Divergent Preference Weights: Even if User 1 and User 2 watch the identical set of movies, differing rating vectors ($R_1 \neq R_2$) scale the binary feature profiles differently. For example, if both users watch Galaxy Mission (Action/SciFi) and Love in Nairobi (Drama/Romance), but User 1 rates them $(5, 1)$ while User 2 rates them $(1, 5)$, User 1's profile will be heavily weighted toward Action/SciFi ($83\%$) while User 2's profile will be weighted toward Drama/Romance ($83\%$).   Divergent Dot-Product Scoring: Because candidate recommendation scores are calculated via vector multiplication $S_j = \mathbf{f}_j \cdot U_{normalized}$, different preference profiles yield different numerical scores across candidate items, resulting in different ranked Top-N recommendations



In [58]:
#GENERATE AND SAVE MOVIES.CSV
import pandas as pd

# 1. Define the original 10 movies dataset (with unique movie 11)
movies_export = pd.DataFrame({
    "movie_id": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11],
    "title": [
        "Galaxy Mission", "Love in Nairobi", "Final Battle", "Family Weekend",
        "Future World", "Broken Hearts", "City Detectives", "Space Warriors",
        "Wedding Weekend", "Hidden Truth", "Nairobi Cyberpunk 2077"
    ],
    "Action":  [1, 0, 1, 0, 1, 0, 1, 1, 0, 0, 1],
    "Comedy":  [0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0],
    "Drama":   [0, 1, 0, 0, 0, 1, 1, 0, 0, 1, 0],
    "Romance": [0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0],
    "SciFi":   [1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1]
})

# 2. Export to CSV (index=False prevents pandas from adding an unnecessary extra index column)
movies_export.to_csv("movies.csv", index=False)

print("movies.csv successfully created!")

movies.csv successfully created!


In [59]:
# Verify by loading movies.csv back into a DataFrame
movies_from_csv = pd.read_csv("movies.csv")
movies_from_csv

,movie_id,title,Action,Comedy,Drama,Romance,SciFi
0,1,Galaxy Mission,1,0,0,0,1
1,2,Love in Nairobi,0,0,1,1,0
2,3,Final Battle,1,0,0,0,0
3,4,Family Weekend,0,1,0,0,0
4,5,Future World,1,0,0,0,1
5,6,Broken Hearts,0,0,1,1,0
6,7,City Detectives,1,0,1,0,0
7,8,Space Warriors,1,0,0,0,1
8,9,Wedding Weekend,0,1,0,1,0
9,10,Hidden Truth,0,0,1,0,0
